In [ ]:
# Pass 1: Intall libraries
!pip install openai PyMuPDF --quiet


In [ ]:
# Pass 2: Librerías necesarias
from openai import OpenAI
import fitz  # PyMuPDF
from getpass import getpass
from google.colab import files

In [ ]:
# Pass 3: Load PFF TICKET
print("📎 Load your ticket in PDF")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]


📎 Sube tu boleta de pago en PDF


Saving PDF-BOLETAEB01-6220609367963[1].pdf to PDF-BOLETAEB01-6220609367963[1] (1).pdf


In [ ]:
# Paso 4: Extract text of PDF
def extraer_texto_pdf(ruta_pdf):
    doc = fitz.open(ruta_pdf)
    texto = ""
    for pagina in doc:
        texto += pagina.get_text()
    doc.close()
    return texto

boleta_texto = extraer_texto_pdf(pdf_path)
boleta_texto

' \n \nGRUPO CEU \nGRUPO CEU SOCIEDAD ANONIMA CERRADA \nCAL. LAS MAGNOLIAS MZA. B LOTE. 8  \nVILLA MARIA DEL TRIUNFO - LIMA - LIMA \nBOLETA DE VENTA ELECTRONICA\nRUC: 20609367963 \nEB01-62 \nFecha de Vencimiento\n:  \nFecha de Emisión\n: 29/01/2023 \nSeñor(es)\n: PAUL ANDRES MELO RAMOS \nDNI \n: 72299843 \nTipo de Moneda\n: SOLES \nObservación\n:  \nCantidad\nUnidad\nMedida\nCódigo\nDescripción\nValor Unitario(*)\nDescuento(*)\nImporte de Venta(**)\nICBPER\n1.00\nUNIDAD\nPE-FA-\n2023\nPROGRAMA DE\nESPECIALIZACION EN\nFINANZAS AVANZADAS\n- DC-MB\n169.49\n0.00\n199.9982\n0.00\nOtros Cargos :\nS/ 0.00 \nOtros Tributos :\nS/0.00 \nICBPER :\nS/ 0.00 \nImporte Total :\nS/200.00 \nSON: DOSCIENTOS Y 00/100 SOLES\n(*) Sin impuestos.\n(**) Incluye impuestos, de ser Op. Gravada.\n \n \n \n \nOp. Gravada :\nS/ 169.49 \nOp. Exonerada :\nS/ 0.00 \nOp. Inafecta :\nS/ 0.00 \nISC :\nS/ 0.00 \nIGV :\nS/ 30.51 \nICBPER :\nS/ 0.00 \nOtros Cargos :\nS/ 0.00 \nOtros Tributos :\nS/ 0.00 \nMonto de Redondeo :

In [ ]:
# Paso 5: API key
api_key = getpass("🔐 Ingresa tu API key de OpenAI: ").strip()
client = OpenAI(api_key=api_key)

🔐 Ingresa tu API key de OpenAI: ··········


In [ ]:
# Paso 6: Prompt para extraer los datos clave de la boleta
prompt_extraccion = f"""
Eres un experto en análisis de boletas electrónicas del Perú.

A partir del siguiente texto extraído de un archivo PDF, quiero que identifiques y devuelvas únicamente los siguientes campos, como **un JSON plano** (sin anidaciones ni listas), donde cada campo sea una clave independiente:

- Fecha de emisión
- RUC del emisor
- Cantidad
- Unidad de medida
- Código
- Descripción
- Valor unitario
- Descuento
- Importe de venta
- IGV
- Importe total

Devuelve **solo** el bloque JSON, sin explicaciones ni texto adicional. No incluyas ninguna lista bajo nombres como "Detalle" o "Items". Cada valor debe estar directamente en el objeto raíz del JSON.

Ejemplo esperado:

{{
  "Fecha de emisión": "dd/mm/aaaa",
  "RUC del emisor": "12345678901",
  "Cantidad": "1.00",
  "Unidad de medida": "UNIDAD",
  ...
}}

Texto de la boleta extraído por OCR:

\"\"\"
{boleta_texto}
\"\"\"
"""

# Paso 7: Extraer los datos con GPT
respuesta_extraccion = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "Eres un analista experto en boletas de venta electrónicas SUNAT."},
        {"role": "user", "content": prompt_extraccion}
    ],
    temperature=0
)

datos_extraidos = respuesta_extraccion.choices[0].message.content.strip()

# Mostrar los datos extraídos
print("📋 Datos extraídos de la boleta:\n")
print(datos_extraidos)

# Paso 8: Análisis estructurado con otro prompt
prompt_analisis = f"""
A partir de estos datos extraídos de una boleta electrónica:

{datos_extraidos}

1. Valida que los campos numéricos sean coherentes (valor unitario + IGV ≈ total).
2. Señala si hay errores o inconsistencias.
3. Describe el tipo de producto o servicio.
4. Evalúa si cumple con lo requerido por SUNAT para un CPE válido.
5. Genera insights tributarios o contables relevantes.
"""

respuesta_analisis = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "Eres un auditor tributario SUNAT especializado en comprobantes electrónicos."},
        {"role": "user", "content": prompt_analisis}
    ],
    temperature=0
)

# Mostrar el análisis final
print("\n🧠 Análisis de la boleta:\n")
print(respuesta_analisis.choices[0].message.content.strip())


📋 Datos extraídos de la boleta:

{
  "Fecha de emisión": "29/01/2023",
  "RUC del emisor": "20609367963",
  "Cantidad": "1.00",
  "Unidad de medida": "UNIDAD",
  "Código": "PE-FA-2023",
  "Descripción": "PROGRAMA DE ESPECIALIZACION EN FINANZAS AVANZADAS - DC-MB",
  "Valor unitario": "169.49",
  "Descuento": "0.00",
  "Importe de venta": "199.9982",
  "IGV": "30.51",
  "Importe total": "200.00"
}

🧠 Análisis de la boleta:

1. Validación de campos numéricos: Según los datos proporcionados, el valor unitario es de 169.49 y el IGV es de 30.51. Si sumamos estos dos valores, obtenemos un total de 200.00, que coincide con el importe total indicado en la boleta electrónica. Por lo tanto, los campos numéricos son coherentes.

2. Errores o inconsistencias: Hay una inconsistencia en el campo "Importe de venta". Según los datos proporcionados, el importe de venta es de 199.9982, lo cual no coincide con la suma del valor unitario y el IGV. Este campo debería ser igual al importe total, es decir, 20

In [ ]:
datos_extraidos

'{\n  "Fecha de emisión": "29/01/2023",\n  "RUC del emisor": "20609367963",\n  "Cantidad": "1.00",\n  "Unidad de medida": "UNIDAD",\n  "Código": "PE-FA-2023",\n  "Descripción": "PROGRAMA DE ESPECIALIZACION EN FINANZAS AVANZADAS - DC-MB",\n  "Valor unitario": "169.49",\n  "Descuento": "0.00",\n  "Importe de venta": "199.9982",\n  "IGV": "30.51",\n  "Importe total": "200.00"\n}'

In [ ]:
# Paso 7.1: Guardar datos extraídos como JSON
import json

# Intentar convertir el texto a JSON (maneja errores si el formato no es válido)
try:
    datos_dict = json.loads(datos_extraidos)
    with open("boleta_extraida.json", "w", encoding="utf-8") as f:
        json.dump(datos_dict, f, ensure_ascii=False, indent=2)
    print("✅ Los datos fueron guardados como 'boleta_extraida.json'")
except Exception as e:
    print("⚠️ Error al guardar el archivo JSON:", e)


✅ Los datos fueron guardados como 'boleta_extraida.json'


In [ ]:
datos_extraidos

'Aquí está la información estructurada en formato JSON:\n\n```json\n{\n  "Fecha de emisión": "29/01/2023",\n  "RUC del emisor": "20609367963",\n  "Detalle": [\n    {\n      "Cantidad": "1.00",\n      "Unidad de medida": "UNIDAD",\n      "Código": "PE-FA-2023",\n      "Descripción": "PROGRAMA DE ESPECIALIZACION EN FINANZAS AVANZADAS - DC-MB",\n      "Valor unitario": "169.49",\n      "Descuento": "0.00",\n      "Importe de venta": "199.9982"\n    }\n  ],\n  "IGV": "30.51",\n  "Importe total": "200.00"\n}\n```\n\nPor favor, ten en cuenta que el OCR puede tener errores y puede que no haya extraído toda la información correctamente. Además, he asumido que el "Importe de venta" incluye el IGV, ya que no se proporcionó un campo separado para el IGV en el detalle del producto.'

In [ ]:
datos_dict


{'Fecha de emisión': '29/01/2023',
 'RUC del emisor': '20609367963',
 'Cantidad': '1.00',
 'Unidad de medida': 'UNIDAD',
 'Código': 'PE-FA-2023',
 'Descripción': 'PROGRAMA DE ESPECIALIZACION EN FINANZAS AVANZADAS - DC-MB',
 'Valor unitario': '169.49',
 'Descuento': '0.00',
 'Importe de venta': '199.9982',
 'IGV': '30.51',
 'Importe total': '200.00'}

In [ ]:
import pandas as pd

# Convertir JSON plano a DataFrame
df = pd.DataFrame([datos_dict])

# Mostrar
df


,Fecha de emisión,RUC del emisor,Cantidad,Unidad de medida,Código,Descripción,Valor unitario,Descuento,Importe de venta,IGV,Importe total
0,29/01/2023,20609367963,1.00,UNIDAD,PE-FA-2023,PROGRAMA DE ESPECIALIZACION EN FINANZAS AVANZA...,169.49,0.00,199.9982,30.51,200.00


## ✅ Conclusion

The conversion of structured data from an electronic receipt in PDF format to flat JSON and then into a DataFrame is a highly efficient and scalable solution for automating accounting and tax processes.

By using language models like GPT, we achieved:

- 📄 **Extracting key information** from unstructured documents (free text or scanned).
- 🧠 **Standardizing required fields** using a robust prompt, avoiding nested structures.
- 📊 **Automatically transforming results** into a `DataFrame` ready for analysis, validation, or export.
- ⚙️ **Reducing human errors** and accelerating repetitive tasks like receipt audits or tax filings.

This approach combines **artificial intelligence + natural language processing + data automation**, creating real impact in areas such as accounting, compliance, tax education, and document management.
